# ViT5 inference — Vietnews test (Kaggle)

**GPU T4 x2 · Internet On · Private · không Add Dataset.**

Notebook tự tải `000001.txt.seg` … `000{N}.txt.seg` từ GitHub `ThanhChinhBK/vietnews` (`data/test_tokenized`). Không upload `test_100` / `test_500`.

Model: `VietAI/vit5-base-vietnews-summarization`  
Generate (khóa): `</s>`, `max_length=256`, `early_stopping=True`, 1 GPU `cuda:0`.

**Thứ tự:** cell pip → **Restart session** → cell `N` + tải file → cell model → cell infer. **Đừng Run All.**

In [ ]:
!nvidia-smi
!pip uninstall -y transformers tokenizers
!pip install -q "transformers==4.44.2" sentencepiece protobuf
import transformers, tokenizers
print("transformers", transformers.__version__, "tokenizers", tokenizers.__version__)

Sau cell trên: **Restart session**. Cell tiếp theo phải in `transformers 4.44.2`. Nếu còn 4.5x / 5.x thì restart chưa xong.

In [ ]:
from pathlib import Path
import json
import urllib.request
from concurrent.futures import ThreadPoolExecutor, as_completed

import torch
import transformers
from transformers import T5Tokenizer, T5ForConditionalGeneration

print("transformers", transformers.__version__)
assert transformers.__version__.startswith("4.44"), "Restart session sau pip, rồi chạy lại cell này"

N = 500
MODEL = "VietAI/vit5-base-vietnews-summarization"
RAW_BASE = "https://raw.githubusercontent.com/ThanhChinhBK/vietnews/master/data/test_tokenized"
DEST = Path("/kaggle/working/test_tokenized")
OUT_PATH = Path("/kaggle/working/preds.json" if N == 100 else f"/kaggle/working/preds_{N}.json")
IDS = [f"{i:06d}.txt.seg" for i in range(1, N + 1)]

def download_one(name):
    out = DEST / name
    if out.exists() and out.stat().st_size > 0:
        return name
    tmp = out.with_suffix(out.suffix + ".part")
    urllib.request.urlretrieve(f"{RAW_BASE}/{name}", tmp)
    tmp.replace(out)
    return name

DEST.mkdir(parents=True, exist_ok=True)
pending = [name for name in IDS if not (DEST / name).exists() or (DEST / name).stat().st_size == 0]
print("cần tải", len(pending), "/", N)
if pending:
    with ThreadPoolExecutor(max_workers=8) as pool:
        futs = [pool.submit(download_one, name) for name in pending]
        done = 0
        for fut in as_completed(futs):
            fut.result()
            done += 1
            if done % 50 == 0 or done == len(pending):
                print("downloaded", done, "/", len(pending))

files = [DEST / name for name in IDS]
missing = [p.name for p in files if not p.exists() or p.stat().st_size == 0]
assert not missing, missing[:5]
assert files[0].name == "000001.txt.seg" and files[-1].name == f"{N:06d}.txt.seg"

def parse_file(path):
    parts = [p.strip() for p in Path(path).read_text(encoding="utf-8").split("\n\n") if p.strip()]
    return {"id": Path(path).name, "title": parts[0], "abstract": parts[1], "body": "\n".join(parts[2:])}

docs = [parse_file(p) for p in files]
print("n files", len(files))
print("first", files[0])
print("last", files[-1])
print("out", OUT_PATH)
print(docs[0]["id"], "body_words", len(docs[0]["body"].split()))

In [ ]:
assert "docs" in dir() and "MODEL" in dir(), "Chạy cell N + tải file trước."

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("device", device)

tok = T5Tokenizer.from_pretrained(MODEL)
model = T5ForConditionalGeneration.from_pretrained(MODEL).to(device)
model.eval()

sample = docs[0]["body"] + "</s>"
enc = tok(sample, return_tensors="pt", truncation=True, max_length=1024)
enc = {k: v.to(device) for k, v in enc.items()}
with torch.no_grad():
    out = model.generate(**enc, max_length=256, early_stopping=True)
print(tok.decode(out[0], skip_special_tokens=True, clean_up_tokenization_spaces=True))

In [ ]:
from pathlib import Path
import json

assert "docs" in dir() and "tok" in dir() and "model" in dir() and "OUT_PATH" in dir(), (
    "Chạy tuần tự sau Restart: tải file → model → cell này. Không Run All, không chạy cell này một mình."
)

preds = json.loads(OUT_PATH.read_text(encoding="utf-8")) if OUT_PATH.exists() else []
done = {row["id"] for row in preds}
print("resume", len(done), "/", len(docs), "→", OUT_PATH)
for doc in docs:
    if doc["id"] in done:
        continue
    text = doc["body"] + "</s>"
    enc = tok(text, return_tensors="pt", truncation=True, max_length=1024)
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.no_grad():
        out = model.generate(**enc, max_length=256, early_stopping=True)
    pred = tok.decode(out[0], skip_special_tokens=True, clean_up_tokenization_spaces=True)
    preds.append({"id": doc["id"], "abstract": doc["abstract"], "pred": pred})
    done.add(doc["id"])
    if len(preds) % 25 == 0:
        OUT_PATH.write_text(json.dumps(preds, ensure_ascii=False), encoding="utf-8")
        print("saved", len(preds))
OUT_PATH.write_text(json.dumps(preds, ensure_ascii=False, indent=2), encoding="utf-8")
print("wrote", OUT_PATH, "n=", len(preds))